# ImageNet-1K formal training: VisionLLaMA + LSSO/RRLSSO

This notebook is the cloud entry point for a single A100 80GB. It streams the gated `timm/imagenet-1k-wds` shards during their first use, writes them atomically to local NVMe, and reuses the cache in later epochs and runs. The training process is launched outside the notebook kernel so a browser disconnect does not stop it.

## 1. Setup
Run from the repository root. Accept the ImageNet terms on Hugging Face before continuing.

In [ ]:
from pathlib import Path
import os, sys, subprocess, getpass, json, shlex

ROOT = Path.cwd().resolve()
assert (ROOT / 'pyproject.toml').is_file(), 'Start Jupyter from the LSSO repository root'
print(ROOT)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[experiments]', 'jupyter', 'nbclient'], check=True)

## 2. Authenticate and configure
The token is entered without echo and is inherited by the detached trainer. It is never written to the notebook or checkpoint.

In [ ]:
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token: ')
assert os.environ['HF_TOKEN'].startswith('hf_'), 'Expected a Hugging Face token'

In [ ]:
CONFIG = {
    'model': 'vision_llama_base_rrlsso_r32',  # change to vision_llama_base_lsso_r32 for the paired run
    'cache_dir': '/local_nvme/imagenet-wds',
    'output': str(ROOT / 'runs/imagenet1k/vision_llama_base_rrlsso_r32'),
    'epochs': 300,
    'batch_size': 256,
    'eval_batch_size': 256,
    'grad_accum': 1,
    'workers': 8,
    'seed': 0,
}
Path(CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

## 3. Checks
Check CUDA, model registration, disk space, and gated repository access before spending GPU time.

In [ ]:
import torch, timm
import examples.models
from huggingface_hub import HfApi

assert torch.cuda.is_available(), 'CUDA is unavailable'
assert CONFIG['model'] in timm.list_models('vision_llama_*')
HfApi(token=os.environ['HF_TOKEN']).dataset_info('timm/imagenet-1k-wds')
disk = os.statvfs(CONFIG['cache_dir'])
free_gb = disk.f_bavail * disk.f_frsize / 2**30
assert free_gb >= 180, f'Only {free_gb:.1f} GiB free; reserve at least 180 GiB'
print(torch.cuda.get_device_name(), f'{free_gb:.1f} GiB cache space free')

In [ ]:
# Optional but strongly recommended before a formal launch: two train and two validation batches.
smoke_output = str(Path(CONFIG['output']).with_name(Path(CONFIG['output']).name + '_smoke'))
smoke = [sys.executable, 'experiments/imagenet_wds_train.py',
         '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
         '--output', smoke_output, '--epochs', '1', '--steps-per-epoch', '2',
         '--max-val-steps', '2', '--batch-size', '8', '--eval-batch-size', '8',
         '--workers', '2', '--seed', str(CONFIG['seed']), '--no-resume']
subprocess.run(smoke, check=True, env=os.environ.copy())

## 4. Launch or resume
The command writes `last.pt`, `best.pt`, `metrics.csv`, and `train.log`. Re-running it resumes from `last.pt`.

In [ ]:
command = [sys.executable, 'experiments/imagenet_wds_train.py',
           '--model', CONFIG['model'], '--cache-dir', CONFIG['cache_dir'],
           '--output', CONFIG['output'], '--epochs', str(CONFIG['epochs']),
           '--batch-size', str(CONFIG['batch_size']),
           '--eval-batch-size', str(CONFIG['eval_batch_size']),
           '--grad-accum', str(CONFIG['grad_accum']),
           '--workers', str(CONFIG['workers']),
           '--seed', str(CONFIG['seed']), '--resume']
log_path = Path(CONFIG['output']) / 'train.log'
log = log_path.open('a', buffering=1)
process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                           env=os.environ.copy(), start_new_session=True)
(Path(CONFIG['output']) / 'trainer.pid').write_text(str(process.pid))
print('pid=', process.pid, 'log=', log_path)

In [ ]:
# Non-blocking status check. Do not keep the notebook attached to the process.
print((Path(CONFIG['output']) / 'trainer.pid').read_text())
if (Path(CONFIG['output']) / 'metrics.csv').exists():
    print((Path(CONFIG['output']) / 'metrics.csv').read_text().splitlines()[-5:])
print('cached shards:', len(list(Path(CONFIG['cache_dir']).glob('*.tar'))))

## 5. Paired experiment
After RRLSSO finishes, change `model` and `output` to `vision_llama_base_lsso_r32` and rerun Sections 3–4. Both runs reuse exactly the same cached ImageNet shards and augmentation recipe. The registered MHA entry is an official-compatible reproduction path, not a required new formal run.